# Python Retest: Combined Test 1 + Test 2

**Duration: 3 hours (180 minutes) · 20 questions · 200 marks · 10 marks each**

This is a fresh retest modeled on the concepts and difficulty of the original Python Test 1 (core Python + pandas) and Test 2 (database-backed ETL). The business examples and datasets are new; the underlying skills are intentionally similar.

## Suggested timing

| Section | Questions | Suggested time |
|---|---:|---:|
| Core Python | Q1–Q9 | 70 minutes |
| Pandas and files | Q10–Q14 | 45 minutes |
| Database and ETL | Q15–Q20 | 65 minutes |
| **Total** | **20** | **180 minutes** |

Each question is worth 10 marks. Use readable functions, handle edge cases, and follow the explicit requirements. Do not load the full 100,000-row event file into memory. The ETL questions depend on earlier cells, just as the original Test 2 did.

## Data

All fixtures are in `seed_data/data/`. The setup cell below locates the folder from either the project root or the `notebooks/` folder. Read each question fully before coding.

In [1]:
from pathlib import Path

# Works whether Jupyter is launched from the project root or notebooks/.
CANDIDATE_DIRS = [Path("seed_data/data"), Path("../seed_data/data")]
DATA_DIR = next(path.resolve() for path in CANDIDATE_DIRS if path.exists())
print("Using data directory:", DATA_DIR)

Using data directory: C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data


## Shared grading scale

- **10 – Excellent:** correct, complete, follows all constraints, handles required edge cases.
- **7 – Good:** main solution correct with one limited omission.
- **5 – Partial:** meaningful attempt but at least one important requirement is incomplete.
- **2 – Poor:** limited understanding or major implementation error.
- **0 – Blank:** no meaningful attempt.

Use one holistic score per question, not separate marks for every bullet.

## Question 1: Deduplicate Client Records by Normalized Email (10 points)

**Concept:** Order-preserving deduplication

Given:

```python
event_labels = ["LOGIN", "EXPORT", "LOGIN", "UPLOAD", "EXPORT", "DELETE"]
```

Write `dedup_preserve_order(values)` that returns a new list containing only the first occurrence of each value, in the original order. Do not modify the input list. Add a short comment explaining why `list(set(values))` does not meet the order requirement.

In [2]:
def dedup_preserve_order(values):
    s = set()
    o = []
    for i  in values:
        if i not in s: 
            s.add(i)
            o.append(i)
    return o

#list(set(values)) does not keep the order of list because sets are unordered

## Question 2: Abstract Delivery Channel Interface (10 points)

**Concept:** Abstract base classes and enforced interfaces

Using `abc`, define an abstract class `DeliveryChannel` with abstract method `channel_name(self)`. Create `EmailChannel` returning `"email"` and `QueueChannel` returning `"queue"`. Demonstrate that `DeliveryChannel()` raises `TypeError`.

In [3]:
from abc import ABC,abstractmethod

class DeliveryChannel(ABC):
    @abstractmethod
    def channel_name(self):
        raise NotImplementedError
class EmailChannel(DeliveryChannel):
    def channel_name(self):
        return "email"
class QueueChannel(DeliveryChannel):
    def channel_name(self):
        return "queue"

print(EmailChannel().channel_name())
print(QueueChannel().channel_name())
try:
    DeliveryChannel()
except Exception as e:
    print(e)

email
queue
Can't instantiate abstract class DeliveryChannel without an implementation for abstract method 'channel_name'


## Question 3: Functional Transformation with map() (10 points)

**Concept:** map() and lambda

Using `map()` and a `lambda`, create a list of squares for integers from 3 through 18 inclusive. Then write one sentence explaining when a list comprehension would be clearer.

In [4]:
sq = list(map(lambda x:x**2,range(3,19)))
print(sq)

lsq = [x**2 for x in range(3,19)]
print(lsq)
# list comprehension require less syntax and easy to comprehend and analyse the condition to impelement

[9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256, 289, 324]
[9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256, 289, 324]


## Question 4: Streaming API Event Metrics (10 points)

**Concept:** Generators and memory-efficient aggregation

`seed_data/data/api_events.csv` contains 100,000 API event rows with columns `event_id, endpoint, status_code, duration_ms, event_time`. Write `read_api_events(path)` as a generator yielding one dictionary per row. Consume it once and calculate, for each endpoint: total valid events, count of 5xx responses, and average duration in milliseconds. Some rows contain non-numeric status or duration values: skip rows that cannot be safely used and count rejected rows. Do not call `list(read_api_events(...))` and do not store all raw rows.

In [5]:
import csv
from collections import defaultdict
def read_api_events(path,stats = None):
    with open(path) as f:
        data = csv.DictReader(f)
        for i in data: 
            try:
                i["status_code"] = int(i["status_code"])
                i["duration_ms"] = float(i["duration_ms"])
                yield i
            except (TypeError,ValueError):
                if stats is not None:stats["rejected"] += 1
                continue
            
stats = {"rejected":0}
d = read_api_events(r"C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data\api_events.csv",stats)
endpoints = defaultdict(lambda : {"total": 0, "5xx": 0, "duration_sum": 0.0})

for i in d:
    ep = endpoints[i["endpoint"]]
    ep["total"] += 1
    ep["duration_sum"] += i["duration_ms"]
    if 500 <= int(i["status_code"]) < 600:
        ep["5xx"] += 1
print("Rejected rows:", stats["rejected"])
for ep, val in endpoints.items():
    avg = val["duration_sum"] / val["total"]
    print(f"{ep}: total={val['total']}, 5xx={val['5xx']}, avg_ms={avg:.2f}")

Rejected rows: 16
/api/customers: total=20008, 5xx=2861, avg_ms=908.98
/api/reports: total=20109, 5xx=2868, avg_ms=902.84
/api/orders: total=19717, 5xx=2711, avg_ms=903.61
/api/products: total=20028, 5xx=2797, avg_ms=903.94
/api/payments: total=20122, 5xx=2781, avg_ms=904.04


## Question 5: Parse Structured API Audit Events with Regex (10 points)

**Concept:** Regular expressions for structured text

`seed_data/data/api_audit.log` contains structured lines such as:

```text
2026-11-01T08:03:00Z INFO actor=U104 action=LOGIN object=dashboard message="user action completed"
```

Using regular expressions only, parse each valid line into a dictionary with `timestamp`, `level`, `actor`, `action`, `object`, and `message`. The quoted message can contain spaces. Skip malformed lines and return both the valid records and the skipped-line count.

In [6]:
import shlex

path = r"C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data\api_audit.log"

c = []
rejected = 0

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        try:
            # Splits into: ['2026-11-01T08:03:00Z', 'INFO', 'actor=U104', 'action=LOGIN', 'object=dashboard', 'message=user action completed']
            parts = shlex.split(line)

            # Turn all 'key=value' items into a dictionary
            kv = dict(item.split("=", 1) for item in parts[2:] if "=" in item)

            c.append({
                "timestamp": parts[0],
                "level": parts[1],
                "actor": kv["actor"],
                "action": kv["action"],
                "object": kv["object"],
                "message": kv["message"],
            })
        except (ValueError, IndexError, KeyError):
            rejected += 1
            continue

print(f"Parsed: {len(c)}, Rejected: {rejected}")
print(c[:3])


Parsed: 149, Rejected: 1
[{'timestamp': '2026-11-01T08:03:00Z', 'level': 'ERROR', 'actor': 'U137', 'action': 'DOWNLOAD', 'object': 'report:monthly', 'message': 'permission check passed'}, {'timestamp': '2026-11-01T08:06:00Z', 'level': 'DEBUG', 'actor': 'U111', 'action': 'LOGIN', 'object': 'report:monthly', 'message': 'user action completed'}, {'timestamp': '2026-11-01T08:09:00Z', 'level': 'DEBUG', 'actor': 'U126', 'action': 'EXPORT', 'object': 'asset:LT-0047', 'message': 'export file generated with warnings'}]


In [7]:
d = '''2026-11-01T08:03:00Z INFO actor=U104 action=LOGIN object=dashboard message="user action completed"'''
print(d.split(' ')[2].split('=')[1])

U104


## Question 6: Repeating-Key Route-Code Encoding (10 points)

**Concept:** Custom transformation functions and alphabet wraparound

Implement `encode_route_code(value)`. The input contains lowercase letters `a-z`. Repeat the shift key `[2, 4, 1]` from left to right: the first character moves 2 positions, the second 4, the third 1, then the key repeats. Wrap around from `z` to `a`. Verify that `encode_route_code("xyz")` returns `"zca"`.

In [8]:
def encode_route_code(value):
    data = [i for i in value]
    sh = [2,4,1]
    ot = []
    for i in data:
        ot.append(chr((ord(i) - ord('a') + sh[data.index(i)])%26 + ord('a')))
    return "".join(i for i in ot)
encode_route_code('xyz') == 'zca'

True

## Question 7: Normalize Asset Tags and Clean Reference Numbers (10 points)

**Concept:** Field normalization and type coercion

Implement both functions:

1. `normalize_asset_tag(value)` converts variants such as `"lt-0047"`, `" LT 0047 "`, `"LT/0047"` and `"pr.1020"` to canonical `"LT-0047"` or `"PR-1020"`. Return `None` for invalid formats. The format is exactly two letters followed by four digits after ignoring spaces, dots, slashes and hyphens.
2. `clean_reference_code(value)` removes every non-digit character and returns the remaining value as an integer, or `None` if there are no digits.

Demonstrate both functions using at least five inputs each.

In [9]:
def normalize_asset_tag(value):
    if value is None:
        return None
    s = str(value).strip()
    if len(s) < 6:
        return None

    prefix = s[:2]
    digits = s[-4:]
    middle = s[2:-4]

    # Check: first 2 are letters, last 4 are digits, and middle only has allowed separators
    if (
        prefix.isalpha()
        and digits.isdigit()
        and all(ch in " ./-" for ch in middle)
    ):
        return f"{prefix.upper()}-{digits}"
    return None


def clean_reference_code(value):
    if value is None:
        return None
    digits = "".join(ch for ch in str(value) if ch.isdigit())
    return int(digits) if digits else None


asset_tags = ["lt-0047", " LT 0047 ", "LT/0047", "pr.1020", "wrong-tag"]
reference_codes = ["WR-00981", " 45020 ", "12/34/56", "#8823", "n/a"]

print([normalize_asset_tag(value) for value in asset_tags])
# Output: ['LT-0047', 'LT-0047', 'LT-0047', 'PR-1020', None]

print([clean_reference_code(value) for value in reference_codes])
# Output: [981, 45020, 123456, 8823, None]


['LT-0047', 'LT-0047', 'LT-0047', 'PR-1020', None]
[981, 45020, 123456, 8823, None]


## Question 8: Refactor File Reading with Logging (10 points)

**Concept:** Specific exception handling and logging

Refactor the following poor pattern into `read_text_files(paths)`:

```python
try:
    data = open(path).read()
except:
    print("failed")
```

Use a context manager and UTF-8. Catch `FileNotFoundError` and `PermissionError` separately. Log each attempted path at DEBUG, successful reads at INFO, and failures at WARNING/ERROR. Never use a bare `except` or silently hide failures. Demonstrate one existing file and one missing path.

In [10]:
import logging

logger = logging.getLogger(__name__)

def read_text_files(paths):
    result = {}
    for path in paths:
        try:
            logger.debug("Debug file read")
            result[path] = open(path).read()
            logger.info("Data read")
        except FileNotFoundError:
            logger.error("File not found")
        except PermissionError:
            logger.error("Permission denied")
    return result

## Question 9: TDD and Unit Tests for Quantity Validation (10 points)

**Concept:** TDD and testable pure functions

Write `parse_positive_quantity(value)`: accept a positive integer or a string containing a positive whole number; reject booleans, zero, negative numbers, decimal strings, blank strings and non-numeric text by raising `ValueError`. Write at least five pytest-style tests (valid and invalid cases). In 3–4 sentences explain red-green-refactor and why this function is easier to test than code mixed with file or database I/O.

In [11]:
%pip install pytest

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Sasiv\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
import pytest
def parse_positive_quantity(value):
    if isinstance(value,bool): raise ValueError("bool")
    elif isinstance(value,int): n = value
    elif isinstance(value,str) & value.strip().isdigit():
        n = int(value.strip())
    else: raise ValueError("wrong no")
    if n <= 0: raise ValueError("below zero")
    return n
assert parse_positive_quantity(3)== 3
assert parse_positive_quantity(' 4') == 4
with pytest.raises(ValueError): parse_positive_quantity(True)
with pytest.raises(ValueError): parse_positive_quantity(0)

## Question 10: Fix Numeric Dtypes in an Operations Survey (10 points)

**Concept:** Pandas missing-value sentinels and dtype coercion

Load `seed_data/data/operations_survey.csv` into a DataFrame. The `years_experience` and `annual_pay` columns contain the literal string `"unknown"` in some rows and therefore load as object dtype. Convert both columns to proper numeric dtypes, changing invalid entries to real missing values. Print the resulting dtypes and missing-value counts.

In [13]:
import pandas as pd
path = r'C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data\operations_survey.csv'

df = pd.read_csv(path)
df["annual_pay"] = pd.to_numeric(df['annual_pay'],errors='coerce')
df['years_experience'] = pd.to_numeric(df['years_experience'],errors='coerce')
df['annual_pay'].isna().sum()
df['years_experience'].isna().sum()

np.int64(13)

## Question 11: Percentage of Remote Workers by Team Size (10 points)

**Concept:** Pandas grouped aggregation and percentages

Using the same survey DataFrame, calculate the percentage of respondents whose `remote_work` value is `"Yes"`, grouped by `team_size`. Return one percentage per team-size bucket. Do not use a hard-coded denominator.

In [14]:
df['remote_work_yes'] = (df["remote_work"] == 'Yes').astype(int)
df.groupby(df["team_size"])["remote_work_yes"].mean().mul(100).round(2)

team_size
Large (500+)       54.25
Medium (51-500)    52.67
Small (1-50)       55.51
Name: remote_work_yes, dtype: float64

## Question 12: Normalize Engagement Type and Report Satisfaction (10 points)

**Concept:** Exact categorical normalization and two-dimensional reporting

Normalize `engagement_type_raw` into exactly three values: `CORE`, `FLEX`, or `OTHER` using a function:

- `permanent`, `staff`, `full time`, `ft`, `employee` → `CORE`.
- `contractor`, `agency`, `freelance`, `vendor` → `FLEX`.
- Missing/blank, multi-value, and every unlisted value → `OTHER`.

Strip whitespace and compare lowercase exact values. Add `engagement_group`, show value counts, then report the percentage distribution of `job_satisfaction` for every `(engagement_group, region)` pair.

In [15]:
def normalize_engagement_type_raw(data):
    if pd.isna(data): return "OTHER"
    value = str(data).strip().lower()
    if value in ['permanent','staff','full time','ft','employee']: return "CORE"
    elif value in ['contactor','agency','freelance','vendor']: return "FLEX"
    else: return "OTHER"
df["engagement_group"] = df["engagement_type_raw"].map(normalize_engagement_type_raw)
df["engagement_type_raw"].value_counts()

pd.crosstab([df["engagement_group"],df["region"]],df["job_satisfaction"],normalize="index").mul(100)

job_satisfaction          Dissatisfied    Neutral  Satisfied  \
engagement_group region                                        
CORE             Central     23.636364  25.454545  21.818182   
                 East        24.444444  24.444444  11.111111   
                 North       19.565217  23.913043  13.043478   
                 South       18.867925  24.528302  18.867925   
                 West        10.204082  24.489796  22.448980   
FLEX             Central     23.333333  23.333333  10.000000   
                 East        20.000000  28.571429   8.571429   
                 North       13.888889  19.444444  36.111111   
                 South       22.222222  20.000000  17.777778   
                 West        20.000000  14.285714  17.142857   
OTHER            Central     14.285714  30.952381  15.476190   
                 East        15.789474  26.315789  17.105263   
                 North       19.696970  19.696970  15.151515   
                 South       16.901408  25.352113  23.943662   
                 West        29.729730  13.513514  20.270270   

job_satisfaction          Very dissatisfied  Very satisfied  
engagement_group region                                      
CORE             Central          18.181818       10.909091  
                 East             20.000000       20.000000  
                 North            10.869565       32.608696  
                 South            20.754717       16.981132  
                 West             24.489796       18.367347  
FLEX             Central          26.666667       16.666667  
                 East             20.000000       22.857143  
                 North            16.666667       13.888889  
                 South            24.444444       15.555556  
                 West             17.142857       31.428571  
OTHER            Central          17.857143       21.428571  
                 East             15.789474       25.000000  
                 North            28.787879       16.666667  
                 South            18.309859       15.492958  
                 West             13.513514       22.972973

## Question 13: Combine All Sheets from a Workbook (10 points)

**Concept:** Multi-sheet Excel processing and schema union

Read every sheet in `seed_data/data/branch_activity.xlsx`. The workbook has one header row per sheet and the sheets do not share identical columns. Add `source_sheet` to each row, combine all sheets into one DataFrame, and write `seed_data/data/branch_activity_combined.csv`. Do not hardcode sheet names or the number of sheets.

In [16]:
import pandas as pd
path = r'C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data\branch_activity.xlsx'

frames = []

df = pd.read_excel(path,sheet_name=None)

for sheet,frame in df.items():
    frame["source_name"] = sheet
    frames.append(frame)
ot = pd.concat(frames,sort=False,ignore_index=True)
ot.to_csv(r'C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data\branch_activity_combined.csv',index=False)


## Question 14: Merge Service Requests and Calculate Client Charges (10 points)

**Concept:** DataFrame merge, derived columns and aggregation

Load `service_requests.csv` (`request_id, client_id, service_code, hours`) and `service_catalog.csv` (`service_code, service_name, hourly_rate`). Merge on `service_code`, calculate `line_amount = hours * hourly_rate`, then calculate total service charges and total hours per client, sorted by total charges descending. Do not mutate the original DataFrames.

In [17]:
p1 = r'C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data\service_requests.csv'
p2 = r'C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data\service_catalog.csv'

p1df = pd.read_csv(p1)
p2df = pd.read_csv(p2)

p3df = pd.merge(p1df,p2df,on="service_code",how="inner")

p3df["line_amount"] = p3df["hours"] * p3df["hourly_rate"]

res = p3df.groupby("client_id").agg(
        total_hours=("hours", "sum"),
        total_line_amount=("line_amount", "sum")).sort_values("total_line_amount", ascending=False)
res


,total_hours,total_line_amount
client_id,,
1032,77,5643.0
1039,69,5008.5
1028,60,4377.5
1007,52,4255.0
1030,57,4212.5
1020,55,3982.0
1033,69,3971.5
1029,65,3873.5
1002,47,3557.0


## Question 15: Config-Driven MySQL Connection and Persistent Logging (10 points)

**Concept:** Configuration, SQLAlchemy and logging

Connect to the `python_retest` MySQL database using `pymysql` via SQLAlchemy. Store host, port, username, password and database in a configuration dictionary or environment variables; do not inline them inside `create_engine`. Configure one logger that writes to both console and `etl_run.log`. Catch and log a connection failure, then re-raise it. Query and display all rows from the seeded `region_lookup` table.

In [18]:
from sqlalchemy import text,create_engine
from sqlalchemy.engine import URL
from sqlalchemy.exc import SQLAlchemyError
import logging

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
db_config = {
    "host":"127.0.0.1",
    "port":3307,
    "user":"root",
    "password":"PythonRetest_2026!",
    "database":"python_retest"
}
url = URL.create(
    drivername="mysql+pymysql",
    host=db_config["host"],
    password=db_config["password"],
    port=db_config["port"],
    database=db_config["database"],
    username=db_config["user"]
)

try:
    engine = create_engine(url)
    with engine.connect() as conn:
        data = conn.execute(
            text('''SELECT country_code, region_name FROM region_lookup''')
        ).fetchall()
    print(data)
except Exception as e:
    logger.error(e)

[('AU', 'Oceania'), ('CA', 'North America'), ('DE', 'Europe'), ('GB', 'Europe'), ('IN', 'South Asia'), ('SG', 'Southeast Asia'), ('US', 'North America'), ('ZA', 'Africa')]


## Question 16: Extract Two Raw Sources Without Cleaning (10 points)

**Concept:** ETL extraction and source preservation

Read `seed_data/data/raw_clients.csv` and `seed_data/data/raw_jobs.json` into two separate Python structures (list of dictionaries or DataFrames). This stage is extraction only: preserve the incomplete customer row and the non-numeric job amount; do not clean or drop anything. Catch missing-file and malformed-JSON errors, log before re-raising, and log the number of rows extracted from each source.

In [19]:
# p1 = r'C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data\raw_clients.csv'
# p2 = r'C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data\raw_jobs.json'
import csv,json
def extract_raw_files(client,jobs):
    try:
        with open(client) as f:
            cdata = list(csv.DictReader(f))
        with open(jobs) as g:
            jdata = json.load(g)
    except FileNotFoundError:
        logger.exception("FNF")
        raise
    except json.JSONDecodeError:
        logger.exception("JDE")
        raise
    logger.info(len(cdata),len(jdata))
    return cdata,jdata
# a,b = extract_raw_files(p1,p2)
# print(len(a))

## Question 17: Clean the Extracted Client and Job Data (10 points)

**Concept:** ETL transformation, numeric coercion and rejection counts

Using the structures from Q16:

- Drop client records whose `client_id` is missing or blank, and log a WARNING with the rejected count.
- Convert job `amount` to numeric; drop rows that cannot be converted and log the affected job ID(s) or count.
- Keep valid rows and report retained and rejected counts for both datasets.

Do not silently modify the raw structures from Q16.

In [20]:
def clean_extracted_data(raw_client,raw_jobs):
    client = pd.DataFrame(raw_client).copy()
    jobs = pd.DataFrame(raw_jobs).copy()
    
    client_before = len(client)
    client["client_id"] = client["client_id"].astype("string").str.strip()
    client = client[client["client_id"].notna() & client["client_id"].ne("")]
    
    jobs_before = len(jobs)
    jobs["amount"] = pd.to_numeric(jobs["amount"],errors="coerce")
    bad_jobs = jobs[jobs["amount"].isna()]
    jobs = jobs.dropna(subset=["amount"])
    
    logger.warning("Dropped %d invalid client rows", client_before - len(client))
    logger.warning("Dropped %d invalid job rows:\n%s", jobs_before - len(jobs), bad_jobs)
    logger.info(
        "Clients: %d -> %d | Jobs: %d -> %d",
        client_before, len(client), jobs_before, len(jobs)
    )
    return client,jobs


## Question 18: Enrich Clients from the MySQL Region Lookup (10 points)

**Concept:** ETL enrichment and reference-data joins

Use the `region_lookup` table queried in Q15 (or query it again). Join it to the cleaned clients on `country_code`, adding `region_name`. Every valid country code in this fixture has a match. Use a left merge or dictionary lookup, verify that no `region_name` is missing, and log the enriched row count. If a required field or lookup is missing, log the error and re-raise it.

In [ ]:
def enrich_clients(cleanc_clint,engine):
    try:
        with engine.connect() as conn:
            lookup = pd.read_sql(text("select country_code,region_name from region_looup"),conn)
            enriched = cleanc_clint.merge(
                        lookup,
                        on="country_code",
                        how="left"
            )
            if enriched["region_name"].isna().any():
                    missing = enriched.loc[enriched["region_name"].isna(),"country_code"]
                    raise ValueError(f"Missing region lookup for country codes: {missing}")
        logger.info("Enriched %d client records", len(enriched))
        return enriched
    except Exception:
        logger.exception("Client enrichment failed")
        raise

## Question 19: Idempotent Load and Round-Trip Validation (10 points)

**Concept:** Safe database loading and read-back verification

Create `clean_clients` in `python_retest` if it does not exist, with `client_id` as the primary key and columns for `name`, `email`, `country_code`, and `region_name`. Load the enriched clients using `INSERT ... ON DUPLICATE KEY UPDATE` keyed on `client_id`, so repeating the same batch does not duplicate rows. Read the table back into a DataFrame. Assert that the row count matches the batch and verify at least one known client's email and region. Log success; log and re-raise database errors.

## Question 20: Refactor the Work into a Testable ETL Pipeline (10 points)

**Concept:** ETL orchestration, modularity and independent testing

Refactor Q16–Q19 into:

- `extract()` — reads both raw files and returns their raw structures.
- `transform(...)` — cleans clients and jobs, queries the region lookup and returns enriched clients plus cleaned jobs.
- `load(...)` — creates/updates `clean_clients` using the same idempotent strategy and validates the result.
- `main()` — calls the stages in order and logs progress before and after each stage.

Call `main()`. Then write 4–6 sentences explaining why independent ETL stages improve testing, debugging and maintenance.

In [22]:
# Your code here